# Model Training (coupler_NCap_cap_matrix)
## Capacitance --> Quantum Metal

Instead of training the inverse model to directly predict Quantum Metal parameters and comparing
against ground truth SQuADDs params, we now chain the inverse model with a frozen surrogate (Qiskit-->Capacitance)
and train on the reconstruction error in capacitance space. This works better because multiple
different Qiskit param sets can produce the same capacitance matrix, and before we werent letting the inverse model explore different parameter sets. By measuring error in capacitance instead, this makes the model only care about if the predicted params give the right desired capacitance values, not whether they match perfectly the exact qiskit data set.

## Configuration

In [1]:
## the parameter file has the hyperparameters
## start there if you want to change the setup

from parameters_surrogate_defined_loss import *

## Library

In [ ]:
import os, gc, joblib, json, time, sys, math, csv

## keep the tensorflow warnings down
## comment these out if you want the warnings back
os.environ['TF_XLA_FLAGS'] = '--tf_xla_enable_xla_devices'
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import tensorflow as tf, platform
tf.keras.backend.set_floatx("float32")

from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy('mixed_float16')

from tensorflow.keras.models import Sequential, load_model, Model
from tensorflow.keras.layers import Input, Dense, Dropout, LeakyReLU
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.regularizers import l2
from tensorflow.python.client import device_lib
from keras_tuner import HyperModel, RandomSearch
from keras_tuner import BayesianOptimization

from pathlib import Path
import numpy as np
import pandas as pd
from pandas import json_normalize
%matplotlib inline
import matplotlib.pyplot as plt
from IPython.display import clear_output
from mpl_toolkits.mplot3d import Axes3D

## imports that come up later
from datetime import datetime
from tensorflow.keras import Sequential

seed = 0

## if the seed stays the same, the random numbers stay the same too
## you get the same random values every run
np.random.seed(seed)

## set the tensorflow seed too
tf.random.set_seed(seed)

## Check GPU

In [ ]:
## check what hardware tensorflow can see
print(device_lib.list_local_devices())

## run !{sys.executable} m pip install u pip
## run !{sys.executable} m pip install u "tensorflow[andcuda]"
## in this cell and restart the kernel.

print(tf.config.list_physical_devices("GPU"))

## check the cuda packages
!{sys.executable} -m pip list | egrep "tensorflow|nvidia-(cuda|cudnn|cublas|nccl)"

## Dataset

### Load

In [ ]:
## load the onehot encoded data (saved from ml_00 notebook)

## inputs are the capacitance values (what we want to invert)
X_train = np.load(f'{DATA_DIR}/npy/x_train_one_hot_encoding_scaled.npy', allow_pickle=True)
X_val   = np.load(f'{DATA_DIR}/npy/x_val_one_hot_encoding_scaled.npy', allow_pickle=True)
X_test  = np.load(f'{DATA_DIR}/npy/x_test_one_hot_encoding_scaled.npy', allow_pickle=True)

## qiskit metal params in onehot encoding
y_train = np.load(f'{DATA_DIR}/npy/y_train_one_hot_encoding_scaled.npy', allow_pickle=True)
y_val   = np.load(f'{DATA_DIR}/npy/y_val_one_hot_encoding_scaled.npy', allow_pickle=True)
y_test  = np.load(f'{DATA_DIR}/npy/y_test_one_hot_encoding_scaled.npy', allow_pickle=True)

## column names
with open(str(Path(METADATA_DIR) / 'X_names'), 'r') as f:
    cap_column_names = f.read().splitlines()

oh_col_names   = np.load(str(Path(METADATA_DIR) / 'y_columns.npy'), allow_pickle=True).astype(str).tolist()
cont_col_names = np.load(str(Path(METADATA_DIR) / 'y_columns_continuous.npy'), allow_pickle=True).astype(str).tolist()
fc_oh_col_names = np.load(str(Path(METADATA_DIR) / 'y_columns_fingers.npy'), allow_pickle=True).astype(str).tolist()

n_continuous = len(cont_col_names)
n_fc_onehot  = len(fc_oh_col_names)
n_oh_total   = n_continuous + n_fc_onehot

print(f'Inputs (Capacitance):     {X_train.shape[1]} columns')
print(f'Outputs (one-hot Qiskit): {y_train.shape[1]} columns ({n_continuous} continuous + {n_fc_onehot} finger_count one-hot)')
print(f'\nCapacitance columns:  {cap_column_names}')
print(f'Continuous Qiskit columns: {cont_col_names}')
print(f'Finger count one-hot columns: {fc_oh_col_names}')

X_train = X_train.astype('float32')
y_train = y_train.astype('float32')
X_val   = X_val.astype('float32')
y_val   = y_val.astype('float32')

### Visualize

In [ ]:
## look at the shapes of training and test sets
print('X_train.shape: ', X_train.shape)
print('X_val.shape:   ', X_val.shape)
print('X_test.shape:  ', X_test.shape)
print('y_train.shape: ', y_train.shape)
print('y_val.shape:   ', y_val.shape)
print('y_test.shape:  ', y_test.shape)
print('\ny_train[0]: ', y_train[0])

display(X_train)
display(y_train)

## look at how it was split and decide if we like the split (we do for now)
total = len(X_train) + len(X_test) + len(X_val)
print('---------------------------------------')  
print('Train set shape x:      {}, {:.2f}%'.format(len(X_train), (len(X_train)*100.)/total))
print('Validation set shape x: {}, {:.2f}%'.format(len(X_val), (len(X_val)*100.)/total))
print('Test set shape x:       {}, {:.2f}%'.format(len(X_test), (len(X_test)*100.)/total))
print('---------------------------------------')

total = len(y_train) + len(y_test) + len(y_val)
print('---------------------------------------')  
print('Train set shape y:      {}, {:.2f}%'.format(len(y_train), (len(y_train)*100.)/total))
print('Validation set shape y: {}, {:.2f}%'.format(len(y_val), (len(y_val)*100.)/total))
print('Test set shape y:       {}, {:.2f}%'.format(len(y_test), (len(y_test)*100.)/total))
print('---------------------------------------')

## bin the input data (capacitance values) and look at distribution

num_cols = X_train.shape[1]
num_rows = math.ceil(num_cols / 3)

fig, axes = plt.subplots(num_rows, 3, figsize=(10, 3 * num_rows))
axes = axes.ravel()

for i in range(num_cols):
    axes[i].hist(X_train[:, i], bins=30, edgecolor='black')
    label = cap_column_names[i] if i < len(cap_column_names) else f'col_{i}'
    axes[i].set_title(f'Training Set {label}', fontsize=9, fontweight='normal')
    axes[i].set_xlabel(label, fontsize=9, fontweight='normal')
    axes[i].set_ylabel('Frequency', fontsize=9, fontweight='normal')

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.savefig(str(Path(PLOTS_DIR) / 'surrogate_loss_training_set_input_distribution.pdf'))
plt.show()

steps_per_epoch = int(np.ceil(len(X_train) / TRAIN_BATCH_SIZE))
LR_DECAY_STEPS = steps_per_epoch * 20   ## decay every ~20 epochs

## MLP

### Create model

Reccomended to download a third party app like "Sleep control Center" or "Amphetamine" to prevent computer from sleeping during the many hour/day long training process

### Make conversion layer

This will take us from one-hot encoded values to the linear version that the surrogate models need. its a custom layer that will not have updatable weights or anything that will mess up training (hopefully)

In [ ]:
## extract finger count values from col names (like 'design_options.finger_count_2' goes to just 2)
fc_values = sorted([int(c.split('_')[-1]) for c in fc_oh_col_names])
print(f'Finger count values: {fc_values}')

## determine the linear encoding column order by reading the original data
## (the surrogate expects columns in the original dataframe order with finger_count as one column)
columns_to_drop = ['design_tool','design_options.prime_width',
    'design_options.prime_gap', 'design_options.second_width',
    'design_options.second_gap', 'design_options.cap_gap_ground',
    'design_options.cap_distance', 'design_options.orientation',
    'design_tool', 'coupler_type']

df_tmp = pd.read_json(str(Path(METADATA_DIR) / 'coupler-NCap-cap_matrix.json'))
y_tmp = json_normalize(df_tmp['design']).drop(columns=columns_to_drop)
linear_col_names = list(y_tmp.columns)
del df_tmp, y_tmp

fc_linear_idx = linear_col_names.index('design_options.finger_count')
n_linear_cols = len(linear_col_names)
print(f'Linear column order ({n_linear_cols} cols): {linear_col_names}')
print(f'finger_count is at position {fc_linear_idx} in the linear order')

## load the finger_count scaler from the surrogate's linear encoding
fc_scaler = joblib.load(str(Path(SCALERS_DIR) / 'scaler_y_linear_design_options.finger_count.save'))
fc_data_min   = float(fc_scaler.data_min_[0])
fc_data_range = float(fc_scaler.data_range_[0])
print(f'finger_count linear scaler: min={fc_data_min}, range={fc_data_range}')

The scalers are different for the version of the data that was used in this notebook vs what the surrogate model expects. We have to take that into account and transform from one to the other in this layer we are building:

In [ ]:
cont_scale_a = np.ones(n_continuous, dtype=np.float32)
cont_scale_b = np.zeros(n_continuous, dtype=np.float32)

for i, col in enumerate(cont_col_names):
    oh_sc  = joblib.load(f'{SCALERS_DIR}/scaler_y_one_hot_{col}.save') ## load the scaler for one hot encoding
    lin_sc = joblib.load(f'{SCALERS_DIR}/scaler_y_linear_{col}.save') ## load scaler used in linear encoding for the surrogate model
    oh_min,  oh_range  = float(oh_sc.data_min_[0]),  float(oh_sc.data_range_[0]) 
    lin_min, lin_range = float(lin_sc.data_min_[0]), float(lin_sc.data_range_[0])
    if lin_range > 0:
        ## save the slope and intercept for this column to use later when going from one hot scaler to linear scaler later
        cont_scale_a[i] = oh_range / lin_range
        cont_scale_b[i] = (oh_min - lin_min) / lin_range

## check if scalers are identical (they should be since same training data)
if np.allclose(cont_scale_a, 1.0) and np.allclose(cont_scale_b, 0.0):
    print('Continuous column scalers are identical between one-hot and linear (as expected)')
else:
    print('WARNING: continuous column scalers differ between one-hot and linear!')
    print(f'  scale_a range: [{cont_scale_a.min():.6f}, {cont_scale_a.max():.6f}]')
    print(f'  scale_b range: [{cont_scale_b.min():.6f}, {cont_scale_b.max():.6f}]')

## load and freeze the surrogate model so its not updated during training
SURROGATE_PATH = str(Path(MODEL_DIR) / 'best_keras_model_model2_surrogate.keras')
surrogate = load_model(SURROGATE_PATH, compile=False)

## freeze it
surrogate.trainable = False
for layer in surrogate.layers:
    layer.trainable = False

surr_input_dim = surrogate.input_shape[-1]
assert surr_input_dim == n_linear_cols, (
    f'Surrogate expects {surr_input_dim} inputs but linear encoding has {n_linear_cols} columns'
)
print(f'\nSurrogate loaded & frozen: {SURROGATE_PATH} ({surrogate.count_params():,} params, 0 trainable)')
print(f'Surrogate input dim: {surr_input_dim}  (matches linear encoding: {n_linear_cols})')

## build the custom layer to go from one hot encoded outputs to linear inputs for the surrogate model
## its made of fixed affine transforms and a weighted sum.
## gradients flow through it because everything is differentiable.

class OneHotToLinearLayer(tf.keras.layers.Layer):
    def __init__(self, n_continuous, fc_values, fc_data_min, fc_data_range,
             fc_linear_idx, cont_scale_a, cont_scale_b, **kwargs):
        kwargs.setdefault('trainable', False)  ## default to False, but don't override if already set
        super().__init__(**kwargs)
        self._n_cont    = n_continuous
        self._fc_vals   = tf.constant([fc_values], dtype=tf.float32)    ## (1, n_fc)
        self._fc_min    = tf.constant(fc_data_min, dtype=tf.float32)
        self._fc_range  = tf.constant(fc_data_range, dtype=tf.float32)
        self._fc_idx    = fc_linear_idx
        self._scale_a   = tf.constant(cont_scale_a, dtype=tf.float32)   ## (n_cont,)
        self._scale_b   = tf.constant(cont_scale_b, dtype=tf.float32)   ## (n_cont,)
        ## store raw values for get_config
        self._cfg = dict(n_continuous=n_continuous, fc_values=fc_values,
                         fc_data_min=fc_data_min, fc_data_range=fc_data_range,
                         fc_linear_idx=fc_linear_idx,
                         cont_scale_a=list(cont_scale_a),
                         cont_scale_b=list(cont_scale_b))

    def call(self, inputs):
        continuous = inputs[:, :self._n_cont]
        fc_onehot  = inputs[:, self._n_cont:]
        
        ## cast constants to match input dtype (mixed precision sends float16)
        fc_vals  = tf.cast(self._fc_vals, inputs.dtype)
        fc_min   = tf.cast(self._fc_min, inputs.dtype)
        fc_range = tf.cast(self._fc_range, inputs.dtype)
        scale_a  = tf.cast(self._scale_a, inputs.dtype)
        scale_b  = tf.cast(self._scale_b, inputs.dtype)
        
        ## convert onehot to single finger_count value via weighted sum
        fc_raw = tf.reduce_sum(fc_onehot * fc_vals, axis=1, keepdims=True)
        
        ## scale to match the surrogate's linear encoding scaler
        fc_scaled = (fc_raw - fc_min) / fc_range
        
        ## rescale continuous columns from onehot scaler space to linear scaler space
        cont_rescaled = continuous * scale_a + scale_b
        
        ## insert finger_count at the correct position in linear column order
        return tf.concat([
            cont_rescaled[:, :self._fc_idx],
            fc_scaled,
            cont_rescaled[:, self._fc_idx:]
        ], axis=1)

    def get_config(self):
        config = super().get_config()
        config.update(self._cfg)
        return config

## build the conversion layer instance
oh_to_linear = OneHotToLinearLayer(
    n_continuous=n_continuous,
    fc_values=fc_values,
    fc_data_min=fc_data_min,
    fc_data_range=fc_data_range,
    fc_linear_idx=fc_linear_idx,
    cont_scale_a=cont_scale_a,
    cont_scale_b=cont_scale_b,
    name='oh_to_linear'
)

## quick sanity test
test_input = tf.constant(y_train[:3].astype('float32'))
test_output = oh_to_linear(test_input)
print(f'Conversion test: input shape {test_input.shape} -> output shape {test_output.shape}')
print(f'Expected output shape: (3, {n_linear_cols})')

## range penalty loss
## penalizes the inverse model for predicting qiskit metal values outside
## the training data range [0, 1] (since data is MinMaxScaled).
## this prevents the model from finding "adversarial" inputs to the surrogate
## that look good in capacitance space but don't correspond to real designs.

PENALTY_WEIGHT = 0.1  ## tune this increase if params are still wild, decrease if reconstruction suffers

def qiskit_range_penalty(y_true_dummy, y_pred_oh):
    """Penalize predictions outside [0, 1] range.
    y_true_dummy is ignored (we pass zeros as dummy targets).
    y_pred_oh is the inverse model's raw output in one-hot scaled space."""
    below = tf.nn.relu(-y_pred_oh)          ## positive when < 0
    above = tf.nn.relu(y_pred_oh - 1.0)     ## positive when > 1
    return tf.reduce_mean(below ** 2 + above ** 2)

## prebuild dummy targets for training (same shape as y, filled with zeros ignored by penalty loss)
dummy_y_train = np.zeros_like(y_train)
dummy_y_val   = np.zeros_like(y_val)
dummy_y_test  = np.zeros_like(y_test)

print(f'Penalty weight: {PENALTY_WEIGHT}')
print(f'Dummy targets shape: {dummy_y_train.shape}')

### Create Model by Hand

In [ ]:
## just checkin to make sure everything looks good still, we want float32
print(X_train.dtype, X_train.shape)
print(y_train.dtype, y_train.shape)

print("TF:", tf.__version__)
print("OS:", platform.platform())
print("Built with CUDA:", tf.test.is_built_with_cuda())
print("Built with ROCm:", tf.test.is_built_with_rocm())
print("GPUs:", tf.config.list_physical_devices("GPU"))

if not KERAS_TUNER and not SWEEP_PARAM_NUM:
    ## model shape string for file naming
    model_shape = f'surrogate_loss_mlp_{X_train.shape[1]}_'
    model_shape += '_'.join(str(l) for l in NEURONS_PER_LAYER)
    model_shape += f'_{y_train.shape[1]}'

if not KERAS_TUNER and not SWEEP_PARAM_NUM:
    ## build inverse model that takes capacitance > onehot encoded qiskit metal params
    inverse_model = Sequential(name='inverse_model')
    inverse_model.add(Input(shape=(X_train.shape[1],), name='cap_input'))
    
    for i, n in enumerate(NEURONS_PER_LAYER):
        inverse_model.add(Dense(n, name='fc{}'.format(i),
                        kernel_initializer='lecun_uniform',
                        kernel_regularizer=tf.keras.regularizers.l2(1e-4)))
        inverse_model.add(LeakyReLU(negative_slope=0.01, name='leaky_relu{}'.format(i)))
        inverse_model.add(Dropout(rate=TRAIN_DROPOUT_RATE, name='dropout{}'.format(i)))
    
    ## output layer predicts all onehot encoded qiskit params
    inverse_model.add(Dense(y_train.shape[1], activation='linear', name='qiskit_output_oh',
                    kernel_initializer='lecun_uniform'))
    
    ## twooutput combined model
    ## output 1 reconstructed capacitance (through frozen surrogate) main loss
    ## output 2 raw qiskit params (onehot) range penalty loss
    combined_input = Input(shape=(X_train.shape[1],), name='combined_input')
    predicted_qiskit_oh = inverse_model(combined_input)
    predicted_qiskit_linear = oh_to_linear(predicted_qiskit_oh)  ## differentiable conversion
    reconstructed_cap = surrogate(predicted_qiskit_linear)
    model = Model(
        inputs=combined_input,
        outputs=[reconstructed_cap, predicted_qiskit_oh],
        name='combined_model'
    )
    
    print('Inverse model (outputs one-hot encoded Qiskit params):')
    inverse_model.summary()
    print('\nCombined model (two outputs: cap reconstruction + Qiskit params for penalty):')
    model.summary()

if not KERAS_TUNER and not SWEEP_PARAM_NUM:
    lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
        initial_learning_rate=LR_INITIAL,  
        decay_steps=LR_DECAY_STEPS,        
        decay_rate=LR_DECAY_RATE,          
        staircase=LR_STAIRCASE             
    )
    
    ## two losses reconstruction + range penalty on qiskit params
    model.compile(
        optimizer=tf.optimizers.Adam(learning_rate=lr_schedule),  
        loss=[TRAIN_LOSS, qiskit_range_penalty],
        loss_weights=[1.0, PENALTY_WEIGHT],
        metrics={model.output_names[0]: [TRAIN_LOSS]}
    )

if not KERAS_TUNER and not SWEEP_PARAM_NUM:
    Path(MODEL_DIR).mkdir(parents=True, exist_ok=True)
    best_model_file = str(Path(MODEL_DIR) / '{}_best_model.keras').format(model_shape)
    last_model_file = str(Path(MODEL_DIR) / '{}_last_model.keras').format(model_shape)

Enable training (`train_and_save`) to overwrite the model file.

In [27]:
train_and_save = True

We use Adam optimizer, minimize the loss specified in parameters, and early stop.

#### Training

In [ ]:
if not KERAS_TUNER and not SWEEP_PARAM_NUM:
    class TrainingPlot(tf.keras.callbacks.Callback):
        def on_train_begin(self, logs={}):
            self.losses = []
            self.val_losses = []
            self.logs = []
        
        def on_epoch_end(self, epoch, logs={}):
            self.logs.append(logs)
            self.losses.append(logs.get('loss'))
            self.val_losses.append(logs.get('val_loss'))
            
            if len(self.losses) > 1:
                clear_output(wait=True)
                N = np.arange(0, len(self.losses))
                plt.figure()
                plt.plot(N, self.losses, label='train_loss')
                plt.plot(N, self.val_losses, label='val_loss')
                plt.title('Training Loss [Epoch {}]'.format(epoch))
                plt.xlabel('Epoch #')
                plt.ylabel('Loss')
                plt.legend()
                plt.show()

class LearningRateMonitor(tf.keras.callbacks.Callback):
    def __init__(self):
        super().__init__()
        self.learning_rates = []

    def _current_lr(self, optimizer):
        lr = getattr(optimizer, 'lr', None) or getattr(optimizer, 'learning_rate', None)
        if isinstance(lr, tf.keras.optimizers.schedules.LearningRateSchedule):
            return float(lr(optimizer.iterations).numpy())
        return float(tf.keras.backend.get_value(lr))

    def on_epoch_end(self, epoch, logs=None):
        try:
            lr_val = self._current_lr(self.model.optimizer)
        except Exception:
            lr_val = float(tf.keras.backend.get_value(self.model.optimizer.learning_rate))
        self.learning_rates.append(lr_val)

%%time

## train the model two targets
## 1. cap reconstruction (input capacitance is the target)
## 2. dummy zeros for the penalty loss (penalty only uses predictions, ignores target)
history = None  
if not KERAS_TUNER and not SWEEP_PARAM_NUM and not SWEEP_DATA_AMOUNT:
    if train_and_save: 
        early_stopping = EarlyStopping(
            monitor='val_loss',
            mode='min',
            patience=TRAIN_EARLY_STOPPING_PATIENCE,
            verbose=1
        )
    
        plot_callback = TrainingPlot()
        lr_monitor = LearningRateMonitor()
        
        model_checkpoint = ModelCheckpoint(
            filepath=best_model_file,          
            monitor='val_loss',
            mode='min',
            save_best_only=True,
            verbose=0
        )

        history = model.fit(
            np.asarray(X_train),
            [np.asarray(X_train), dummy_y_train],    ## [cap target, dummy for penalty]
            epochs=400,                   
            batch_size=TRAIN_BATCH_SIZE,  
            validation_data=(np.asarray(X_val), [np.asarray(X_val), dummy_y_val]),  
            callbacks=[early_stopping, model_checkpoint, plot_callback, lr_monitor],  
            verbose=1
        )
        
        model.save(last_model_file)

Load the saved best model and use it from now on.

In [30]:
if not KERAS_TUNER and not SWEEP_PARAM_NUM and not SWEEP_DATA_AMOUNT:
    model = load_model(best_model_file, custom_objects={
        'OneHotToLinearLayer': OneHotToLinearLayer
    })

### Sweep total number of parameters to find the right range

In [ ]:
if SWEEP_PARAM_NUM:
    qiskit_oh_dim = y_train.shape[1]
    cap_dim = X_train.shape[1]
    
    def build_combined_mlp(neurons_per_layer):
        inv = Sequential(name='inverse_model')
        inv.add(Input(shape=(cap_dim,), name='cap_input'))
        for i, n in enumerate(neurons_per_layer):
            inv.add(Dense(n, name=f'fc{i}', kernel_initializer='lecun_uniform',
                            kernel_regularizer=tf.keras.regularizers.l2(1e-4)))
            inv.add(LeakyReLU(negative_slope=0.01, name=f'leaky_relu{i}'))
            inv.add(Dropout(rate=TRAIN_DROPOUT_RATE, name=f'dropout{i}'))
        inv.add(Dense(qiskit_oh_dim, activation='linear', name='qiskit_output_oh',
                        kernel_initializer='lecun_uniform'))
        combined_input = Input(shape=(cap_dim,))
        qiskit_oh = inv(combined_input)
        qiskit_lin = oh_to_linear(qiskit_oh)
        cap_recon = surrogate(qiskit_lin)
        return Model(combined_input, [cap_recon, qiskit_oh])
    
    def make_optimizer():
        lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
            initial_learning_rate=LR_INITIAL,
            decay_steps=LR_DECAY_STEPS,
            decay_rate=LR_DECAY_RATE,
            staircase=LR_STAIRCASE
        )
        return tf.optimizers.Adam(learning_rate=lr_schedule)
    
    def train_one_config(neurons_per_layer, seed=0):
        tf.keras.backend.clear_session()
        tf.random.set_seed(seed)
        np.random.seed(seed)
        global surrogate, oh_to_linear
        surrogate = load_model(SURROGATE_PATH, compile=False)
        surrogate.trainable = False
        for layer in surrogate.layers:
            layer.trainable = False
        oh_to_linear = OneHotToLinearLayer(
            n_continuous=n_continuous, fc_values=fc_values,
            fc_data_min=fc_data_min, fc_data_range=fc_data_range,
            fc_linear_idx=fc_linear_idx, cont_scale_a=cont_scale_a,
            cont_scale_b=cont_scale_b, name='oh_to_linear')
        combined = build_combined_mlp(neurons_per_layer)
        combined.compile(optimizer=make_optimizer(),
                        loss=[TRAIN_LOSS, qiskit_range_penalty],
                        loss_weights=[1.0, PENALTY_WEIGHT])
        early_stopping = EarlyStopping(
            monitor='val_loss', mode='min',
            patience=TRAIN_EARLY_STOPPING_PATIENCE, verbose=0,
            restore_best_weights=True)
        history = combined.fit(
            np.asarray(X_train), [np.asarray(X_train), dummy_y_train],
            validation_data=(np.asarray(X_val), [np.asarray(X_val), dummy_y_val]),
            epochs=400, batch_size=TRAIN_BATCH_SIZE,
            callbacks=[early_stopping], verbose=0)
        best_val = min(history.history['val_loss'])
        return combined, history, best_val

if SWEEP_PARAM_NUM:
    run_id = datetime.now().strftime('%Y%m%d_%H%M%S')
    out_dir = 'sweep_outputs'
    os.makedirs(out_dir, exist_ok=True)
    csv_path = os.path.join(out_dir, f'surrogate_loss_sweep_results_{run_id}.csv')
    df.to_csv(csv_path, index=False)
    print('Saved:', csv_path)

if SWEEP_PARAM_NUM:
    plt.figure()
    plt.scatter(df['total_params'], df['best_val_loss'])
    plt.xscale('log')
    plt.xlabel('Total parameters (log scale)')
    plt.ylabel('Best val_loss')
    plt.title('Surrogate Loss: Best val_loss vs model size')
    plt.savefig(str(Path(PLOTS_DIR) / 'surrogate_loss_params_vs_loss.png'))
    plt.show()

### Sweep amount of data used in training

In [34]:
if SWEEP_DATA_AMOUNT:
    FIXED_DEPTH = 1          
    FIXED_WIDTH = 1024        
    FIXED_NEURONS = [FIXED_WIDTH] * FIXED_DEPTH
    TRAIN_FRACTIONS = np.linspace(0.3, 1, 20)  
    SWEEP_SEEDS = [0, 1, 2, 3, 4]  

    def make_subset(X, y, frac, seed):
        assert 0 < frac <= 1.0
        n = len(X)
        m = max(1, int(np.floor(frac * n)))
        rng = np.random.default_rng(seed)
        idx = rng.choice(n, size=m, replace=False)
        return X[idx], y[idx], m

### Keras Tuner to Find Best Hyperparameters

Run this if you want to use keras tuner to make the model rather than doing it by hand

In [ ]:

if KERAS_TUNER and not SWEEP_PARAM_NUM:
    cap_dim = X_train.shape[1]
    qiskit_oh_dim = y_train.shape[1]
    
    def build_hypermodel(hp):
        tf.keras.backend.clear_session()
        gc.collect()
        
        ## reload surrogate + rebuild conversion layer after clear_session
        surr = load_model(SURROGATE_PATH, compile=False)
        surr.trainable = False
        for layer in surr.layers:
            layer.trainable = False
        converter = OneHotToLinearLayer(
            n_continuous=n_continuous, fc_values=fc_values,
            fc_data_min=fc_data_min, fc_data_range=fc_data_range,
            fc_linear_idx=fc_linear_idx, cont_scale_a=cont_scale_a,
            cont_scale_b=cont_scale_b, name='oh_to_linear')
        
        n_layers = hp.Int('n_layers', min_value=1, max_value=4, default=2)
        neurons_per_layer = [hp.Int(f'neurons_{i}', min_value=64, max_value=1024, step=64) for i in range(n_layers)]
        dropout_rate = hp.Float('dropout_rate', 0.0, 0.3, step=0.05)
        l2_reg = hp.Float('l2_reg', 1e-6, 1e-2, sampling='LOG', default=1e-4)
        lr_initial = hp.Float('learning_rate', 1e-3, 1e-1, sampling='LOG', default=1e-2)
        use_batchnorm = hp.Boolean('use_batchnorm', default=True)
        penalty_wt = hp.Float('penalty_weight', 0.01, 1.0, sampling='LOG', default=0.1)
        
        ## build the inverse model (outputs onehot encoded qiskit params)
        inv = Sequential(name='inverse_model')
        inv.add(Input(shape=(cap_dim,), name='cap_input'))
        for i, n_units in enumerate(neurons_per_layer):
            inv.add(Dense(n_units, name=f'fc{i}', kernel_initializer='he_normal',
                            kernel_regularizer=tf.keras.regularizers.l2(l2_reg)))
            if use_batchnorm:
                inv.add(tf.keras.layers.BatchNormalization(name=f'bn{i}'))
            inv.add(LeakyReLU(negative_slope=0.01, name=f'leaky_relu{i}'))
            inv.add(Dropout(rate=dropout_rate, name=f'dropout{i}'))
        inv.add(Dense(qiskit_oh_dim, name='qiskit_output_oh', kernel_initializer='he_normal'))
        
        ## twooutput model cap reconstruction + raw qiskit params for penalty
        combined_input = Input(shape=(cap_dim,), name='combined_input')
        qiskit_oh = inv(combined_input)
        qiskit_lin = converter(qiskit_oh)
        cap_recon = surr(qiskit_lin)
        combined = Model(combined_input, [cap_recon, qiskit_oh], name='combined_model')
        combined.compile(
            optimizer=tf.optimizers.Adam(learning_rate=lr_initial),
            loss=[TRAIN_LOSS, qiskit_range_penalty],
            loss_weights=[1.0, penalty_wt],
        )
        return combined

if KERAS_TUNER and not SWEEP_PARAM_NUM:
    tuner = BayesianOptimization(
        build_hypermodel,
        objective='val_loss',
        max_trials=KERAS_TUNER_TRIALS,
        executions_per_trial=2,
        directory=KERAS_DIR + '/hyper_tuning_surrogate_defined_loss_penalty_added',
        project_name='surrogate_loss_mlp_tuning_bayesian',
    )

if KERAS_TUNER and not SWEEP_PARAM_NUM:
    early_stopping = EarlyStopping(
        monitor='val_loss', mode='min',
        patience=TRAIN_EARLY_STOPPING_PATIENCE, verbose=1
    )
    reduce_lr = ReduceLROnPlateau(
        monitor='val_loss', factor=0.5,
        patience=max(10, TRAIN_EARLY_STOPPING_PATIENCE // 3),
        min_lr=1e-6, verbose=1
    )

if KERAS_TUNER and not SWEEP_PARAM_NUM:
    tuner.search(
        np.asarray(X_train), [np.asarray(X_train), dummy_y_train],
        epochs=400, batch_size=TRAIN_BATCH_SIZE,
        validation_data=(np.asarray(X_val), [np.asarray(X_val), dummy_y_val]),
        callbacks=[early_stopping, reduce_lr],
        verbose=1
    )

encoding = 'surrogate_defined_loss'
if KERAS_TUNER and not SWEEP_PARAM_NUM:
    os.makedirs(MODEL_DIR, exist_ok=True)
    best_model_file = f'{MODEL_DIR}/best_keras_model_{encoding}.keras'
    best_combined = tuner.get_best_models(1)[0]
    best_combined.save(best_model_file)
    
    ## also save just the inverse model by itself for later use
    inverse_only = best_combined.get_layer('inverse_model')
    inverse_only.save(f'{MODEL_DIR}/best_inverse_model_{encoding}.keras')
    print(f'Saved combined model: {best_model_file}')
    print(f'Saved inverse-only model: model/best_inverse_model_{encoding}.keras')
    
    tf.keras.backend.clear_session()
    gc.collect()
    with tf.device('/CPU:0'):
        loaded_model = load_model(best_model_file, compile=False,
                                  custom_objects={'OneHotToLinearLayer': OneHotToLinearLayer})

### View the model

In [ ]:
if KERAS_TUNER and not SWEEP_PARAM_NUM:
    print('Combined model (inverse + conversion + frozen surrogate):')
    best_combined.summary()
    print('\nInverse model only:')
    inverse_only.summary()

if not KERAS_TUNER and not SWEEP_PARAM_NUM:
    print('\n---- Model Summary ----')
    model.summary()

### Evaluation

Plot training history.

### Visualize gradients for best model

In [ ]:
if KERAS_TUNER and not SWEEP_PARAM_NUM and VISUALIZE_GRADIENTS:
    class GradientNormLogger(tf.keras.callbacks.Callback):
        def __init__(self, x_probe, y_probe, layer_name_prefixes=('fc', 'output'),
                     log_every=1, max_items_per_prefix=10, verbose=1):
            super().__init__()
            self.x_probe = tf.convert_to_tensor(x_probe)
            self.y_probe = tf.convert_to_tensor(y_probe)
            self.layer_name_prefixes = tuple(layer_name_prefixes)
            self.log_every = int(log_every)
            self.max_items_per_prefix = int(max_items_per_prefix)
            self.verbose = int(verbose)
            self.records = []
        def _want_layer(self, var_name):
            return any(var_name.startswith(pfx) for pfx in self.layer_name_prefixes)
        def on_epoch_end(self, epoch, logs=None):
            logs = logs or {}
            if (epoch + 1) % self.log_every != 0: return
            with tf.GradientTape() as tape:
                y_pred = self.model(self.x_probe, training=True)
                loss = self.model.compiled_loss(self.y_probe, y_pred)
            grads = tape.gradient(loss, self.model.trainable_weights)
            rec = {'epoch': int(epoch + 1), 'probe_loss': float(loss.numpy())}
            per_layer = {}
            for w, g in zip(self.model.trainable_weights, grads):
                if g is None: continue
                wname = w.name.split(':')[0]
                if not self._want_layer(wname): continue
                g_norm = tf.linalg.global_norm([g]).numpy().item()
                rec[f'grad_norm__{wname}'] = float(g_norm)
                layer_key = wname.split('/')[0]
                per_layer.setdefault(layer_key, []).append(g_norm)
            for layer_key, norms in per_layer.items():
                rec[f'grad_mean__{layer_key}'] = float(np.mean(norms))
                rec[f'grad_max__{layer_key}'] = float(np.max(norms))
            g_all = [g for g in grads if g is not None]
            rec['grad_global_norm'] = float(tf.linalg.global_norm(g_all).numpy().item()) if g_all else float('nan')
            self.records.append(rec)
            for k in list(rec.keys()):
                if k != 'epoch': logs[k] = rec[k]
        def to_csv(self, path):
            import csv
            if not self.records: return
            keys = sorted({k for r in self.records for k in r.keys()})
            with open(path, 'w', newline='') as f:
                w = csv.DictWriter(f, fieldnames=keys); w.writeheader(); w.writerows(self.records)

if KERAS_TUNER and not SWEEP_PARAM_NUM and VISUALIZE_GRADIENTS:
    probe_n = min(256, len(X_train))
    x_probe = np.asarray(X_train[:probe_n])
    y_probe = [np.asarray(X_train[:probe_n]), np.zeros((probe_n, y_train.shape[1]), dtype='float32')]
    grad_logger = GradientNormLogger(x_probe=x_probe, y_probe=y_probe,
        layer_name_prefixes=('fc0', 'qiskit_output_oh'), log_every=1, verbose=1)
    best_hp = tuner.get_best_hyperparameters(1)[0]
    model = tuner.hypermodel.build(best_hp)
    lr_monitor = LearningRateMonitor()
    history = model.fit(
        np.asarray(X_train), [np.asarray(X_train), dummy_y_train],
        epochs=400, batch_size=TRAIN_BATCH_SIZE,
        validation_data=(np.asarray(X_val), [np.asarray(X_val), dummy_y_val]),
        callbacks=[early_stopping, lr_monitor, grad_logger], verbose=1)
    grad_logger.to_csv(f'{PLOTS_DIR}/{encoding}_gradients.csv')
    del model; tf.keras.backend.clear_session(); gc.collect()

if KERAS_TUNER and not SWEEP_PARAM_NUM and VISUALIZE_GRADIENTS:
    dfg = pd.DataFrame(grad_logger.records)
    plt.figure()
    plt.plot(dfg['epoch'], dfg['grad_global_norm'])
    plt.yscale('log')
    plt.title('Gradient tracking for a single batch across epochs')
    plt.xlabel('Epoch'); plt.ylabel('Gradient magnitude (log scale)')
    plt.tight_layout(); plt.savefig(f'{PLOTS_DIR}/{encoding}_grad_global_norm.pdf'); plt.show()
    for lk in ['fc0', 'qiskit_output_oh']:
        col = f'grad_mean__{lk}'
        if col in dfg.columns:
            plt.figure(); plt.plot(dfg['epoch'], dfg[col]); plt.yscale('log')
            plt.title(f'Gradient Mean Norm: {lk} (probe batch)')
            plt.xlabel('Epoch'); plt.ylabel('Mean grad norm (log scale)')
            plt.tight_layout(); plt.savefig(f'{PLOTS_DIR}/{encoding}_grad_mean_{lk}.pdf'); plt.show()

### Look at best model

In [ ]:
if KERAS_TUNER and not SWEEP_PARAM_NUM and not VISUALIZE_GRADIENTS:
    best_hp = tuner.get_best_hyperparameters(1)[0]
    model = tuner.hypermodel.build(best_hp)
    lr_monitor = LearningRateMonitor()
    history = model.fit(
        np.asarray(X_train), [np.asarray(X_train), dummy_y_train],
        epochs=400, batch_size=TRAIN_BATCH_SIZE,
        validation_data=(np.asarray(X_val), [np.asarray(X_val), dummy_y_val]),
        callbacks=[early_stopping, lr_monitor], verbose=1)
    del model; tf.keras.backend.clear_session(); gc.collect()

plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('Surrogate-defined loss (total: reconstruction + range penalty)')
plt.ylabel('Loss'); plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='best')
plt.tight_layout(); plt.savefig(f'{PLOTS_DIR}/surrogate_loss_history.pdf'); plt.show()

plt.plot(lr_monitor.learning_rates)
plt.title('Learning Rate over Epochs')
plt.xlabel('Epoch'); plt.ylabel('Learning Rate')
plt.tight_layout(); plt.savefig(f'{PLOTS_DIR}/surrogate_loss_learning_rate.pdf'); plt.show()

### Test set evaluation

In [ ]:
## evaluate on test set
tf.keras.backend.clear_session()
gc.collect()

def get_loss(eval_out):
    if isinstance(eval_out, dict):
        return float(eval_out.get('loss', list(eval_out.values())[0]))
    if isinstance(eval_out, (list, tuple, np.ndarray)):
        return float(eval_out[0])
    return float(eval_out)

combined_model = load_model(best_model_file, compile=False,
    custom_objects={'OneHotToLinearLayer': OneHotToLinearLayer})
combined_model.compile(optimizer='adam',
    loss=[TRAIN_LOSS, qiskit_range_penalty],
    loss_weights=[1.0, PENALTY_WEIGHT])
eval_result = combined_model.evaluate(
    np.asarray(X_test), [np.asarray(X_test), dummy_y_test])
total_loss = float(eval_result[0])
cap_recon_loss = float(eval_result[1])
penalty_loss = float(eval_result[2])

print(f'Total test loss: {total_loss}')
print(f'  Reconstruction loss ({TRAIN_LOSS}): {cap_recon_loss}')
print(f'  Range penalty loss: {penalty_loss} (weighted: {penalty_loss * PENALTY_WEIGHT})')

## also evaluate how well the inverse model predicts qiskit params directly
inverse_model = combined_model.get_layer('inverse_model')
qiskit_oh_pred = inverse_model.predict(np.asarray(X_test), verbose=0)
qiskit_mse = float(np.mean((np.asarray(y_test) - qiskit_oh_pred) ** 2))
print(f'Qiskit param MSE in one-hot space (for reference): {qiskit_mse}')

## check how many predictions are out of [0,1] range
out_of_range = np.sum((qiskit_oh_pred < 0) | (qiskit_oh_pred > 1))
total_values = qiskit_oh_pred.size
print(f'Out-of-range values: {out_of_range}/{total_values} ({100*out_of_range/total_values:.1f}%)')

results_df = pd.DataFrame([{
    'Model': 'Inverse (Surrogate-Defined Loss + Range Penalty)',
    'Train Loss Metric': TRAIN_LOSS,
    'Reconstruction Loss': cap_recon_loss,
    'Penalty Loss': penalty_loss,
    'Penalty Weight': PENALTY_WEIGHT,
    'Total Loss': total_loss,
    'Qiskit Param MSE (OH)': qiskit_mse,
    'Out of Range %': 100*out_of_range/total_values,
}])
print(results_df.to_string(index=False))
results_df.to_csv(str(Path(RESULTS_DIR) / 'training/surrogate_loss_test_results.csv'), index=False)
test_loss_result = cap_recon_loss

## Compare predictions vs. test set

In [ ]:
csv_data = [[
    DATA_AUGMENTATION,
    'surrogate_loss_model',
    'InverseModel_SurrogateLoss',
    test_loss_result,
    TRAIN_LOSS,
    TRAIN_DROPOUT_RATE,
    TRAIN_EARLY_STOPPING_PATIENCE,
    TRAIN_BATCH_SIZE,
    '0.15/0.15',
    LR_INITIAL,
    LR_DECAY_STEPS,
    LR_DECAY_RATE,
    LR_STAIRCASE
]]

csv_file = str(Path(RESULTS_DIR) / 'training/surrogate_loss_results/training/history_losses.csv')
if not os.path.exists(csv_file):
    with open(csv_file, 'w') as file:
        file.write('data_augmentation,model_shape,model_type,test_loss,train_loss,train_dropout_rate,'
                   'train_early_stop_patience,train_batch_size,train_val_split,lr_initial,'
                   'lr_decay_step,lr_decay_rate,lr_stair_case\n')
with open(csv_file, mode='a', newline='') as file:
    writer = csv.writer(file)
    writer.writerows(csv_data)
df = pd.read_csv(csv_file)
styled_df = df.style.apply(lambda s: ['color: red' if v else '' for v in s], subset=['test_loss'])
display(styled_df)

## predictaroo on test set
tf.keras.backend.clear_session()
gc.collect()

with tf.device('/CPU:0'):
    combined_model = load_model(best_model_file, compile=False,
        custom_objects={'OneHotToLinearLayer': OneHotToLinearLayer})
    ## twooutput model [cap_reconstructed, qiskit_oh_predicted]
    predictions = combined_model.predict(np.asarray(X_test), verbose=0)
    if isinstance(predictions, list):
        cap_reconstructed = predictions[0]
        qiskit_oh_predicted = predictions[1]
    else:
        ## fallback for singleoutput model
        cap_reconstructed = predictions
        inverse_model = combined_model.get_layer('inverse_model')
        qiskit_oh_predicted = inverse_model.predict(np.asarray(X_test), verbose=0)

## look at how well the reconstructed capacitance matches the input capacitance

X_test_cur = np.asarray(X_test)
y_test_cur = np.asarray(y_test)  ## ground truth qiskit params in onehot (for reference)
cap_recon  = np.asarray(cap_reconstructed)
qiskit_pred = np.asarray(qiskit_oh_predicted)

n_samples, n_cap_cols = X_test_cur.shape
n_oh_cols = qiskit_pred.shape[1]
n_samples_to_show = 3

## reconstruction errors (capacitance space this is what we trained on)
cap_abs_errors = np.abs(X_test_cur - cap_recon)

print('capacitance reconstruction for the loss')
for i in range(n_samples_to_show):
    rows = []
    for j in range(n_cap_cols):
        label = cap_column_names[j] if j < len(cap_column_names) else f'cap_col_{j}'
        rows.append({'param': label, 'ref': X_test_cur[i,j], 'pred': cap_recon[i,j],
                     'abs_error': cap_abs_errors[i,j]})
    print(f'- Sample {i} - Capacitance reconstruction')
    print(pd.DataFrame(rows).to_string(index=False))
    print()
print('Cap reconstruction error stats (scaled):')
print('  min:', float(cap_abs_errors.min()), ' median:', float(np.median(cap_abs_errors)),
      ' max:', float(cap_abs_errors.max()))

### Unscaled test vs predictions

In [ ]:
## unscale everything and look at errors in real units that we can actually make sense of
with open(str(Path(METADATA_DIR) / 'X_names'), 'r') as f:
    cap_names = f.read().splitlines()
qiskit_names = np.load(str(Path(METADATA_DIR) / 'y_columns.npy'), allow_pickle=True).astype(str).tolist()
## unscale input capacitance using X scalers
## (check which scaler prefix is available)
x_scaler_prefix = 'scaler_X_one_hot' if os.path.exists(f'{SCALERS_DIR}/scaler_X_one_hot_{cap_names[0]}.save') else 'scaler_X_linear'
X_test_unscaled = np.asarray(X_test_cur.copy())
for i in range(X_test_unscaled.shape[0]):
    for j in range(X_test_unscaled.shape[1]):
        cap_name = cap_names[j] if j < len(cap_names) else f'col_{j}'
        scaler = joblib.load(f'{SCALERS_DIR}/{x_scaler_prefix}_{cap_name}.save')
        X_test_unscaled[i, j] = scaler.inverse_transform([[X_test_unscaled[i, j]]])[0][0]
## unscale reconstructed capacitance using same X scalers
cap_recon_unscaled = np.asarray(cap_recon.copy())
for i in range(cap_recon_unscaled.shape[0]):
    for j in range(cap_recon_unscaled.shape[1]):
        cap_name = cap_names[j] if j < len(cap_names) else f'col_{j}'
        scaler = joblib.load(f'{SCALERS_DIR}/{x_scaler_prefix}_{cap_name}.save')
        cap_recon_unscaled[i, j] = scaler.inverse_transform([[cap_recon_unscaled[i, j]]])[0][0]
## unscale qiskit param predictions (onehot encoded)
qiskit_pred_unscaled = np.asarray(qiskit_pred.copy())
y_test_unscaled = np.asarray(y_test_cur.copy())
for i in range(qiskit_pred_unscaled.shape[0]):
    for j in range(qiskit_pred_unscaled.shape[1]):
        col_name = qiskit_names[j] if j < len(qiskit_names) else f'col_{j}'
        ## onehot finger_count columns are already 0/1, no scaler to invert
        if col_name.startswith('design_options.finger_count_'):
            continue
        scaler = joblib.load(f'{SCALERS_DIR}/scaler_y_one_hot_{col_name}.save')
        qiskit_pred_unscaled[i, j] = scaler.inverse_transform([[qiskit_pred_unscaled[i, j]]])[0][0]
        y_test_unscaled[i, j] = scaler.inverse_transform([[y_test_unscaled[i, j]]])[0][0]
n_samples_to_show = 3
cap_abs_unscaled = np.abs(X_test_unscaled - cap_recon_unscaled)

print('unscaled capacitance reconstruction for the loss')
for i in range(n_samples_to_show):
    ## capacitance reconstruction table
    rows = []
    for j in range(X_test_unscaled.shape[1]):
        rows.append({'param': cap_names[j], 'ref_unscaled': X_test_unscaled[i,j],
                     'pred_unscaled': cap_recon_unscaled[i,j], 'abs_error': cap_abs_unscaled[i,j]})
    print(f'- Sample {i} (Unscaled) - Capacitance reconstruction')
    print(pd.DataFrame(rows).to_string(index=False))

    ## predicted qiskit params that produced this capacitance
    print(f'\n ------ Predicted Quantum Metal params for sample {i}: -------')
    for j, col_name in enumerate(qiskit_names):
        if col_name.startswith('design_options.finger_count_'):
            continue  ## skip individual onehot columns, we'll decode them below
        pred_val = qiskit_pred_unscaled[i, j]
        ref_val  = y_test_unscaled[i, j]
        short_name = col_name.replace('design_options.', '')
        print(f'    {short_name:30s}  pred={pred_val}  ref={ref_val}  err={abs(pred_val - ref_val):}')

    ## decode onehot finger_count to single predicted value
    fc_cols = [c for c in qiskit_names if c.startswith('design_options.finger_count_')]
    fc_indices = [qiskit_names.index(c) for c in fc_cols]
    fc_pred_oh = qiskit_pred_unscaled[i, fc_indices]
    fc_ref_oh  = y_test_unscaled[i, fc_indices]
    predicted_fc = fc_values[np.argmax(fc_pred_oh)]
    true_fc      = fc_values[np.argmax(fc_ref_oh)]
    print(f'    {"finger_count":30s}  pred={predicted_fc}  ref={true_fc}')
    print()

print('Unscaled cap reconstruction error stats:')
print('  min:', float(cap_abs_unscaled.min()), ' median:', float(np.median(cap_abs_unscaled)),
      ' max:', float(cap_abs_unscaled.max()))